# Диалоговые модели: decoder-only и encoder-decoder

Два независимых эксперимента. В каждом своя модель и свой датасет. Общие только функции предобработки и подсчёта метрик.

Для каждой архитектуры:

1. Выбрать модель: https://huggingface.co/models
2. Выбрать датасет: https://huggingface.co/datasets
3. Проверить модель в сценарии, близком к её датасету, в zero-shot режиме.
4. Посчитать BLEU и METEOR исходной модели на валидационном сабсете.
5. Дообучить модель на обучающем сабсете.
6. Проверить модель в том же сценарии после дообучения.
7. Посчитать BLEU и METEOR дообученной модели на валидационном сабсете.
8. Сравнить метрики дообученной и исходной моделей.

Идентификаторы моделей и датасетов ниже — плейсхолдеры.

Зависимости: `torch`, `transformers`, `datasets`, `sacrebleu`, `nltk`, `pandas`, `tqdm`.


In [ ]:
import os

import nltk
import pandas as pd
import sacrebleu
import torch
from datasets import Dataset, load_dataset
from nltk.translate.meteor_score import meteor_score
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


## Плейсхолдеры

`layout`:

- `"turns"` — реплики в одной колонке: список строк или список словарей с ролью и текстом. Роли `user` / `human` — запрос, `assistant` / `system` / `bot` — ответ. Если ролей нет, реплики чередуются, первая — пользователь.
- `"columns"` — запрос и ответ в двух отдельных колонках.


In [ ]:
# --- decoder-only ---
DECODER_MODEL = "<decoder-only>"          # https://huggingface.co/models
DECODER_DATASET = "<dataset>"             # https://huggingface.co/datasets
DECODER_DATASET_CONFIG = None
DECODER_LAYOUT = "turns"                  # "turns" | "columns"
DECODER_DIALOGUE_COLUMN = "<dialogue>"
DECODER_CONTEXT_COLUMN = "<context>"
DECODER_RESPONSE_COLUMN = "<response>"

# --- encoder-decoder ---
ENCDEC_MODEL = "<encoder-decoder>"        # https://huggingface.co/models
ENCDEC_DATASET = "<dataset>"              # https://huggingface.co/datasets
ENCDEC_DATASET_CONFIG = None
ENCDEC_LAYOUT = "turns"                   # "turns" | "columns"
ENCDEC_DIALOGUE_COLUMN = "<dialogue>"
ENCDEC_CONTEXT_COLUMN = "<context>"
ENCDEC_RESPONSE_COLUMN = "<response>"
ENCDEC_PREFIX = ""

TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"
VAL_SIZE = 0.1

SEED = 42
EPOCHS = 1
LR = 5e-5
BATCH_SIZE = 4
GRAD_ACCUM = 2
EVAL_BATCH_SIZE = 8

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 64
MAX_NEW_TOKENS = 64
MAX_HISTORY_TURNS = 6
NUM_BEAMS = 4

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
N_QUAL_EXAMPLES = 3

set_seed(SEED)


## Загрузка пар «запрос — ответ»

Контекст — последние реплики диалога, заканчивающиеся запросом пользователя. Цель — ответ системы. Если у датасета нет валидационного сплита, он отрезается от обучающего.


In [ ]:
USER_ROLES = {"user", "human", "customer", "usr", "speaker1"}
SYSTEM_ROLES = {"system", "assistant", "bot", "agent", "gpt", "speaker2"}


def turn_role_and_text(turn):
    if isinstance(turn, str):
        return None, turn.strip()
    if isinstance(turn, dict):
        role = str(turn.get("role") or turn.get("speaker") or turn.get("from") or "").strip().lower()
        text = (
            turn.get("content")
            or turn.get("text")
            or turn.get("value")
            or turn.get("utterance")
            or ""
        )
        if role in USER_ROLES:
            role = "user"
        elif role in SYSTEM_ROLES:
            role = "system"
        else:
            role = None
        return role, str(text).strip()
    return None, str(turn).strip()


def format_context(turns):
    lines = []
    for role, text in turns[-MAX_HISTORY_TURNS:]:
        speaker = "User" if role == "user" else "System"
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)


def pairs_from_dialogue(dialogue):
    if isinstance(dialogue, str):
        dialogue = [line for line in dialogue.splitlines() if line.strip()]
    parsed = [turn_role_and_text(turn) for turn in dialogue]
    parsed = [(role, text) for role, text in parsed if text]
    if not parsed:
        return []

    if not any(role is not None for role, _ in parsed):
        parsed = [("user" if i % 2 == 0 else "system", text) for i, (_, text) in enumerate(parsed)]

    pairs = []
    history = []
    pending_user = None
    for role, text in parsed:
        if role == "user":
            pending_user = text
            continue
        if role == "system" and pending_user:
            context = format_context(history + [("user", pending_user)])
            pairs.append({"context": context, "response": text})
            history.append(("user", pending_user))
            history.append(("system", text))
            pending_user = None
    return pairs


def pairs_from_columns(example, context_column, response_column):
    context = example[context_column]
    response = example[response_column]
    if isinstance(context, list):
        context = " ".join(str(part) for part in context)
    if isinstance(response, list):
        response = " ".join(str(part) for part in response)
    context = str(context).strip()
    response = str(response).strip()
    if not context or not response:
        return []
    return [{"context": f"User: {context}", "response": response}]


def take_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return dataset.shuffle(seed=SEED).select(range(max_samples))


def load_dialogue(dataset_name, dataset_config, layout, dialogue_column, context_column, response_column):
    raw = load_dataset(dataset_name, dataset_config) if dataset_config else load_dataset(dataset_name)
    print(raw)
    print("columns:", raw[TRAIN_SPLIT].column_names)
    print("example:", raw[TRAIN_SPLIT][0])

    if TRAIN_SPLIT not in raw:
        raise ValueError(f"Нет сплита {TRAIN_SPLIT!r}. Доступны: {list(raw)}")
    if VAL_SPLIT in raw:
        train_raw, val_raw = raw[TRAIN_SPLIT], raw[VAL_SPLIT]
    else:
        parts = raw[TRAIN_SPLIT].train_test_split(test_size=VAL_SIZE, seed=SEED)
        train_raw, val_raw = parts["train"], parts["test"]

    def build_pairs(split_ds):
        contexts, responses = [], []
        for example in tqdm(split_ds, desc="pairs"):
            try:
                if layout == "columns":
                    pairs = pairs_from_columns(example, context_column, response_column)
                elif layout == "turns":
                    pairs = pairs_from_dialogue(example[dialogue_column])
                else:
                    raise ValueError("layout должен быть 'turns' или 'columns'")
            except KeyError as error:
                raise KeyError(
                    "Не найдена колонка диалога. Заполните имена колонок под выбранный датасет."
                ) from error
            for pair in pairs:
                contexts.append(pair["context"])
                responses.append(pair["response"])
        if not contexts:
            raise ValueError("Пары «запрос — ответ» не собраны. Проверьте layout и имена колонок.")
        return Dataset.from_dict({"context": contexts, "response": responses})

    train_ds = take_subset(build_pairs(train_raw), MAX_TRAIN_SAMPLES)
    val_ds = take_subset(build_pairs(val_raw), MAX_VAL_SAMPLES)
    print("train pairs:", len(train_ds), "val pairs:", len(val_ds))
    print(train_ds[0])
    return train_ds, val_ds


## Обучение и метрики

- **Decoder-only.** Промпт — контекст и суффикс `System:`. Loss считается только по токенам ответа.
- **Encoder-decoder.** В энкодер подаётся контекст, декодер учится генерировать ответ.

BLEU — корпусной sacreBLEU (0–100). METEOR — среднее по валидационным ответам (0–1). Сопоставление синонимов в NLTK рассчитано на английский.

Качественная проверка берёт несколько запросов из валидации этой же модели.


In [ ]:
def prepare_tokenizer(tokenizer, model):
    added = 0
    if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
        added = tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    if added:
        model.resize_token_embeddings(len(tokenizer))
    if tokenizer.pad_token_id is not None:
        model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "right"
    return tokenizer, model


def encode_decoder_prompt(tokenizer, context):
    cue_ids = tokenizer("\nSystem:", add_special_tokens=False)["input_ids"]
    limit = max(1, MAX_SOURCE_LENGTH - len(cue_ids))
    ids = tokenizer(context, add_special_tokens=True, truncation=True, max_length=limit)["input_ids"]
    eos_id = tokenizer.eos_token_id
    if eos_id is not None and ids and ids[-1] == eos_id:
        ids = ids[:-1]
    return ids + cue_ids


def tokenize_decoder(batch, tokenizer):
    input_ids, labels, attention_mask = [], [], []
    eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    for context, response in zip(batch["context"], batch["response"]):
        prompt_ids = encode_decoder_prompt(tokenizer, context)
        response_ids = tokenizer(
            response,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
        )["input_ids"]
        ids = prompt_ids + response_ids + eos
        input_ids.append(ids)
        labels.append([-100] * len(prompt_ids) + response_ids + eos)
        attention_mask.append([1] * len(ids))
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


def tokenize_encoder_decoder(batch, tokenizer):
    sources = [ENCDEC_PREFIX + context for context in batch["context"]]
    model_inputs = tokenizer(sources, truncation=True, max_length=MAX_SOURCE_LENGTH)
    labels = tokenizer(text_target=batch["response"], truncation=True, max_length=MAX_TARGET_LENGTH)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def pad_batch(sequences, pad_id, side, device):
    width = max(len(seq) for seq in sequences)
    input_ids, attention_mask = [], []
    for seq in sequences:
        pad_len = width - len(seq)
        if side == "left":
            input_ids.append([pad_id] * pad_len + seq)
            attention_mask.append([0] * pad_len + [1] * len(seq))
        else:
            input_ids.append(seq + [pad_id] * pad_len)
            attention_mask.append([1] * len(seq) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, device=device),
        "attention_mask": torch.tensor(attention_mask, device=device),
    }


def generate_responses(model, tokenizer, contexts, architecture):
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device
    predictions = []
    for start in tqdm(range(0, len(contexts), EVAL_BATCH_SIZE), desc="generate", leave=False):
        batch = contexts[start : start + EVAL_BATCH_SIZE]
        if architecture == "decoder":
            sequences = [encode_decoder_prompt(tokenizer, context) for context in batch]
            inputs = pad_batch(sequences, tokenizer.pad_token_id, "left", device)
            prompt_width = inputs["input_ids"].shape[1]
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    num_beams=NUM_BEAMS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            decoded = tokenizer.batch_decode(output[:, prompt_width:], skip_special_tokens=True)
        else:
            sources = [ENCDEC_PREFIX + context for context in batch]
            sequences = tokenizer(sources, truncation=True, max_length=MAX_SOURCE_LENGTH)["input_ids"]
            inputs = pad_batch(sequences, tokenizer.pad_token_id, "right", device)
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    num_beams=NUM_BEAMS,
                    do_sample=False,
                )
            decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        predictions.extend(text.strip() for text in decoded)
    return predictions


def generation_metrics(predictions, references):
    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    scores = []
    for prediction, reference in zip(predictions, references):
        pred_tokens = nltk.word_tokenize(prediction) or [""]
        ref_tokens = nltk.word_tokenize(reference) or [""]
        scores.append(meteor_score([ref_tokens], pred_tokens))
    meteor = sum(scores) / len(scores)
    return {"bleu": bleu, "meteor": meteor}


def show_examples(title, model, tokenizer, contexts, references, architecture):
    print(f"\n{title}")
    predictions = generate_responses(model, tokenizer, contexts, architecture)
    for index, (context, reference, prediction) in enumerate(zip(contexts, references, predictions), 1):
        print(f"--- {index} ---")
        print(context)
        print("Reference:", reference)
        print("Model:", prediction)
    return predictions


def compare_metrics(model_name, original, finetuned):
    frame = pd.DataFrame(
        [
            {
                "model": model_name,
                "metric": metric,
                "original": original[metric],
                "finetuned": finetuned[metric],
                "delta": finetuned[metric] - original[metric],
            }
            for metric in ("bleu", "meteor")
        ]
    )
    print("\n8. Сравнение метрик")
    for row in frame.itertuples(index=False):
        if row.delta > 0:
            direction = "выше"
        elif row.delta < 0:
            direction = "ниже"
        else:
            direction = "на том же уровне, что и"
        print(
            f"{row.metric}: дообученная модель {direction} исходной "
            f"({row.original:.4f} -> {row.finetuned:.4f}, delta={row.delta:+.4f})"
        )
    return frame


def run_experiment(model_name, architecture, train_ds, val_ds):
    if architecture == "decoder":
        model_cls = AutoModelForCausalLM
        tokenize = tokenize_decoder
    elif architecture == "encoder-decoder":
        model_cls = AutoModelForSeq2SeqLM
        tokenize = tokenize_encoder_decoder
    else:
        raise ValueError("architecture должен быть 'decoder' или 'encoder-decoder'")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = model_cls.from_pretrained(model_name)
    tokenizer, model = prepare_tokenizer(tokenizer, model)

    qual_n = min(N_QUAL_EXAMPLES, len(val_ds))
    qual_contexts = val_ds["context"][:qual_n]
    qual_references = val_ds["response"][:qual_n]

    show_examples("3. Zero-shot", model, tokenizer, qual_contexts, qual_references, architecture)

    print("\n4. BLEU и METEOR исходной модели")
    original_predictions = generate_responses(model, tokenizer, val_ds["context"], architecture)
    original_metrics = generation_metrics(original_predictions, val_ds["response"])
    print(original_metrics)

    print("\n5. Дообучение")
    tokenized_train = train_ds.map(
        lambda batch: tokenize(batch, tokenizer),
        batched=True,
        remove_columns=train_ds.column_names,
    )
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    args = TrainingArguments(
        output_dir=f"./enc_dec_runs/{architecture}",
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
        logging_steps=50,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
        seed=SEED,
        overwrite_output_dir=True,
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
    )
    trainer.train()
    model = trainer.model

    show_examples("6. После дообучения", model, tokenizer, qual_contexts, qual_references, architecture)

    print("\n7. BLEU и METEOR дообученной модели")
    finetuned_predictions = generate_responses(model, tokenizer, val_ds["context"], architecture)
    finetuned_metrics = generation_metrics(finetuned_predictions, val_ds["response"])
    print(finetuned_metrics)

    comparison = compare_metrics(model_name, original_metrics, finetuned_metrics)

    del trainer
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return comparison


## Decoder-only

Свой датасет, шаги 3–8. Этот блок не читает данные encoder-decoder эксперимента.


In [ ]:
decoder_train, decoder_val = load_dialogue(
    DECODER_DATASET,
    DECODER_DATASET_CONFIG,
    DECODER_LAYOUT,
    DECODER_DIALOGUE_COLUMN,
    DECODER_CONTEXT_COLUMN,
    DECODER_RESPONSE_COLUMN,
)
decoder_metrics = run_experiment(DECODER_MODEL, "decoder", decoder_train, decoder_val)
decoder_metrics


## Encoder-decoder

Свой датасет, шаги 3–8. Этот блок не читает данные decoder-only эксперимента.


In [ ]:
encdec_train, encdec_val = load_dialogue(
    ENCDEC_DATASET,
    ENCDEC_DATASET_CONFIG,
    ENCDEC_LAYOUT,
    ENCDEC_DIALOGUE_COLUMN,
    ENCDEC_CONTEXT_COLUMN,
    ENCDEC_RESPONSE_COLUMN,
)
encdec_metrics = run_experiment(ENCDEC_MODEL, "encoder-decoder", encdec_train, encdec_val)
encdec_metrics
